In [ ]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-27

### CMIP6 data from Copernicus CDS

**CMIP6 (Coupled Model Intercomparison Project Phase 6)** data are also accessible through the **Copernicus Climate Data Store (CDS)**, which provides climate projections processed and formatted for user-friendly analysis.

#### 💡 **Key characteristics:**

- **Provider:** Copernicus Climate Change Service (C3S), operated by ECMWF.
- **Data source:** Global climate model outputs from CMIP6, including bias-adjusted datasets.
- **Variables available:** Near-surface air temperature (`tas`), precipitation (`pr`), and other essential climate variables.
- **Scenarios:** Historical and future projections based on **SSP scenarios** (e.g., SSP1-2.6, SSP2-4.5, SSP3-7.0, SSP5-8.5).
- **Spatial resolution:** Typically regridded to ~1° x 1° or ~0.5° x 0.5° depending on the dataset.
- **Temporal resolution:** Monthly or daily averages depending on the dataset.
- **Temporal coverage:** Varies by model and scenario; historical data up to ~2014 and projections from 2015 onwards to 2100.

#### 🖥️ **Advantages of using CDS CMIP6 data:**

1. **Bias-adjusted products:** Some datasets are bias-corrected using ERA5 reference data to enhance consistency with observations.
2. **Standardized format:** NetCDF files following CF conventions, ready for direct use in climate impact models.
3. **User-friendly API:** Data can be searched and downloaded using the **Copernicus Climate Data Store API** (`cdsapi`) for automated workflows.

#### 🔗 **Access and download:**

- **CDS Web portal:** [https://cds.climate.copernicus.eu/](https://cds.climate.copernicus.eu/)
- **Direct CMIP6 dataset link:** [https://cds.climate.copernicus.eu/cdsapp#!/dataset/projections-cmip6](https://cds.climate.copernicus.eu/cdsapp#!/dataset/projections-cmip6)


In [ ]:
from pyhydra.utils import interactive_map
from pyhydra.data_sources.climate_change import download_CDS_CMIP6
import os
from pathlib import Path

## 🗺️ Select area of interest

Draw a **rectangle** on the map to define the bounding box. Read `coordinates_list[0]` in the configuration cell below.

In [ ]:
coordinates_list = interactive_map(zoom=4, center=(20, 0))

In [ ]:
# === SETUP WORKING DIRECTORY ===

# Automatically detect the path where the notebook is located
notebook_dir = Path().resolve()
os.chdir(notebook_dir)

# === GENERAL PARAMETERS ===

# CDS API endpoint and your personal API key
url = 'https://cds.climate.copernicus.eu/api'
api_key = 'YOUR_CDS_API_KEY'   # https://cds.climate.copernicus.eu/profile  # Replace with your actual API key

# Temporal resolution of the requested data (can be 'monthly' or 'daily')
temporal_resolution = 'monthly'

# Model to use: can be a specific model name or 'All' to retrieve all available
model = 'All'

# List of climate experiments (historical and future scenarios)
experiments = ['historical', 'ssp2_4_5', 'ssp5_8_5']

# List of variables to download
variables = [
    'precipitation',
    'daily_maximum_near_surface_air_temperature',
    'daily_minimum_near_surface_air_temperature'
]

# Output directory base path
download_base_dir = '/workspace/data/cmip6/'

# Geographic bounding box: [North, West, South, East]
#area = [39.842286, -12.041016, 33.504759, 2.285156]
area = coordinates_list[0] if coordinates_list else [39.84, -12.04, 33.50, 2.29]  # fallback: Iberian Peninsula

# === MODEL SKIP LIST ===

# This set will store any models that previously failed with critical errors
model_skip_list = set()

# === DOWNLOAD LOOP ===

# Skip download if API key not configured
if api_key == "YOUR_CDS_API_KEY":
    print("[CDS] API key not configured — set api_key above to download.")
    print("[CDS] Register at: https://cds.climate.copernicus.eu/profile")
else:
    # Iterate over each variable and experiment combination
    for variable in variables:
        for experiment in experiments:
    
            # Set the start and end dates depending on the experiment
            if experiment == 'historical':
                start_date = '1950-01-01'
                end_date = '2014-12-31'
            else:
                start_date = '2015-01-01'
                end_date = '2099-12-31'
    
            # Create output subdirectory based on variable and scenario
            if variable == 'precipitation':
                download_dir = os.path.join(download_base_dir, 'pr', experiment)
            elif variable == 'daily_maximum_near_surface_air_temperature':
                download_dir = os.path.join(download_base_dir, 'tasmax', experiment)
            elif variable == 'daily_minimum_near_surface_air_temperature':
                download_dir = os.path.join(download_base_dir, 'tasmin', experiment)
    
            os.makedirs(download_dir, exist_ok=True)
    
            # Skip models that are known to produce critical errors
            if model in model_skip_list:
                print(f"⚠️ Skipping model {model} due to previous critical errors.")
                continue
    
            # Call the download function for each combination
            try:
                download_CDS_CMIP6(
                    url=url,
                    api_key=api_key,
                    start_date=start_date,
                    end_date=end_date,
                    temporal_resolution=temporal_resolution,
                    variables=[variable],
                    model=model,
                    experiments=[experiment],
                    area=area,
                    download_base_dir=download_dir,
                    max_workers=5
                )
    
            except RuntimeError as e:
                # Handle critical errors, such as incompatible model-variable combinations
                if "RoocsValueError" in str(e):
                    print(f"❌ Critical error for model {model}: {e}. It will be skipped in future iterations.")
                    model_skip_list.add(model)

[CDS] API key not configured — set api_key above to download.
[CDS] Register at: https://cds.climate.copernicus.eu/profile
